## 🛠️ 0. Colab Environment Setup
Run once per Colab session. Installs pinned PyTorch + Mamba2 wheels and mounts your Drive.


In [ ]:
# Uninstall conflicting Colab defaults
!pip uninstall -y transformers sentence-transformers torch torchvision torchaudio

# Install pinned PyTorch cu121 (must match Mamba wheel builds)
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.39.3 packaging triton==3.0.0

# Pre-built Mamba2 wheels (saves 20-30 min vs compiling from source)
!wget -qO causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!wget -qO mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!pip install causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl
!pip install mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl

# Scientific stack
!pip install yfinance gluonts tqdm utilsforecast pyyaml pandas numpy submitit torchmetrics gpytorch statsforecast properscoring

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 87.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.9 MB/s eta 0:00:00
     ━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ngngsonan/SC-Mamba.git /content/SC-Mamba 2>/dev/null || echo 'Repo already cloned'

%cd /content/SC-Mamba
!git pull


Mounted at /content/drive
Already up to date.


In [ ]:
!git pull

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 9.32 KiB | 1.86 MiB/s, done.
From https://github.com/ngngsonan/SC-Mamba
   cabe1bd..5c59449  main       -> origin/main
Updating cabe1bd..5c59449
Fast-forward
 benchmark/01_train_single_dataset.py |   85 +--
 core/train.py                        |   20 +
 logs/log_error.txt                   | 1001 ++++++++++++++++++++++++++++------
 3 files changed, 895 insertions(+), 211 deletions(-)


In [ ]:
import os, sys
# ── Critical env vars for Triton 3.x on T4/A100 ─────────────────────────────
# TRITON_F32_DEFAULT=ieee prevents IndexError: map::at on Turing GPUs.
# Must be set BEFORE any mamba_ssm or triton import.
os.environ['TRITON_F32_DEFAULT'] = 'ieee'
os.environ['SC_MAMBA_DIAG'] = '1'

PROJECT_ROOT = '/content/SC-Mamba'
CKPT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f"✅ CWD = {os.getcwd()}")

# Sanity-check key files
for path in ['core/train.py', 'core/models.py', 'core/real_data_val_pipeline.py', 'core/real_data_args.yaml']:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"  {status}  {path}")


✅ CWD = /content/SC-Mamba
  ✅  core/train.py
  ✅  core/models.py
  ✅  core/real_data_val_pipeline.py
  ✅  core/real_data_args.yaml


# Single training N>1
N>1, as number_assests based on characteristic of each dataset

In [ ]:
# @title 01_train_single_N_mnulti.py
# ============================================================
# 🚀 SC-Mamba — Cross-Asset Causal Graph Training
# Trigger: num_assets > 1 + real_train_datasets → auto-activate
#          MultivariateRealDataset (train from scratch, no finetune)
# ============================================================
import yaml, os, subprocess
from pathlib import Path

PROJECT_ROOT   = '/content/SC-Mamba'
CHECKPOINT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

# ── Danh sách các thí nghiệm Multivariate ───────────────────
EXPERIMENTS = {
    'exchange_rate': {
        'num_assets'        : 8,     # Exchange Rate: đúng 8 đồng tiền
        'pred_len'          : 20,
        'prior_mix_frac'    : 0.5,   # 50% synthetic GP + 50% real aligned
        'batch_size'        : 8,     # B×N = 8×8 = 64 series/step (T4-safe)
        'beta_kl'           : 0.2,   # (Giảm từ 0.5 xuống 0.2 để tránh Gradient Vanishing cho tau)
        'beta_anneal_epochs': 10,    # (Kéo dài warmup)
        'gamma_sparsity'    : 0.1,   # L1 penalty on mask density → sparse causal graph
        'training_rounds'   : 200,
        'sub_day'           : False, # Daily freq → no hour/minute features
    },
}

# ── Template cấu hình (shared across experiments) ───────────
BASE_CONFIG = {
    # Core settings
    'seed'                 : 42,
    'debugging'            : False,
    'scaler'               : 'min_max',
    'sin_pos_enc'          : False,
    'sin_pos_const'        : False,
    'encoding_dropout'     : 0.1,
    'handle_constants_model': True,

    # Backbone architecture (Mamba2 — matches existing checkpoints)
    'ssm_config': {
        'bidirectional'         : False,
        'enc_conv'              : True,
        'init_dil_conv'         : True,
        'enc_conv_kernel'       : 5,
        'init_conv_kernel'      : 5,
        'init_conv_max_dilation': 3,
        'global_residual'       : False,
        'in_proj_norm'          : False,
        'initial_gelu_flag'     : True,
        'linear_seq'            : 15,
        'mamba2'                : True,
        'norm'                  : True,
        'norm_type'             : 'layernorm',
        'num_encoder_layers'    : 2,
        'd_state'               : 128,
        'residual'              : False,
        'token_embed_len'       : 1024,
        'block_expansion'       : 2,
        'chunk_size'            : 256,
        'headdim'               : 128,
    },

    # Training schedule
    'num_epochs'        : 30, #30,   # Train from scratch — epoch 0
    'validation_rounds' : 50,
    'real_test_interval': 5,

    # Learning rate
    'lr_scheduler' : 'cosine',
    'initial_lr'   : 5e-5,
    'learning_rate': 1e-5,      # (Tăng từ 1e-7 lên 1e-5 để kích thích tau học lại)
    't_max'        : -1,        # Auto-set to num_epochs

    # Context/prediction length
    'context_len'      : 256,
    'min_seq_len'      : 60,
    'max_seq_len'      : 256,
    'pred_len_sample'  : False, # Fixed pred_len (stable Cross-Asset windows)
    'pred_len_min'     : 10,

    # Loss
    'loss'             : 'nll',
    'multipoint'       : True,
    'sample_multi_pred': 0.3,
    'diag_prints'      : True,
    'nll_detach'       : True,      # Prevent Variance Starvation (detached sigma2/mu grads)
    'spectral_config'  : {
        'tau_init'     : 1.0,       # Initial Gumbel-Softmax temperature
        'alpha_init'   : 10.0,      # Initial mask density (log_alpha ≈ 2.3)
    },

    # Checkpoint
    'model_prefix'    : CHECKPOINT_DIR,
    'wandb'           : False,
    'continue_training': False, # ← Train từ epoch 0 (không load checkpoint cũ)

    # Synthetic prior settings (shared structure, mix_frac set per experiment)
    'prior_config': {
        'curriculum_learning'  : False,
        'mixup_prob'           : 0.0,
        'mixup_series'         : 4,
        'damp_and_spike'       : False,
        'damping_noise_ratio'  : 0.0,
        'spike_noise_ratio'    : 0.0,
        'spike_signal_ratio'   : 0.0,
        'spike_batch_ratio'    : 0.0,
        'fp_options': {
            'linear_random_walk_frac': 0,
            'seasonal_only'  : False,
            'scale_noise'    : [0.6, 0.3],
            'trend_exp'      : False,
            'harmonic_scale_ratio': 0.4,
            'harmonic_rate'  : 0.75,
            'trend_additional': True,
            'transition_ratio': 0.0,
        },
        'gp_prior_config': {
            'max_kernels'            : 6,
            'likelihood_noise_level' : 0.4,
            'noise_level'            : 'random',
            'use_original_gp'        : False,
            'gaussians_periodic'     : True,
            'peak_spike_ratio'       : 0.1,
            'subfreq_ratio'          : 0.2,
            'periods_per_freq'       : 0.5,
            'gaussian_sampling_ratio': 0.2,
            'kernel_periods'         : [4, 5, 7, 21, 24, 30, 60, 120],
            'max_period_ratio'       : 1.0,
            'kernel_bank': {
                'matern_kernel'         : 1.5,
                'linear_kernel'         : 1,
                'periodic_kernel'       : 5,
                'polynomial_kernel'     : 0,
                'spectral_mixture_kernel': 0,
            },
        },
    },
}


for dataset_name, exp_cfg in EXPERIMENTS.items():
    print(f"\n{'='*65}")
    print(f"🚀 CROSS-ASSET TRAINING: {dataset_name.upper()}")
    print(f"   num_assets    : {exp_cfg['num_assets']}")
    print(f"   prior_mix_frac: {exp_cfg['prior_mix_frac']}  "
          f"(synthetic {exp_cfg['prior_mix_frac']*100:.0f}% + "
          f"real {(1-exp_cfg['prior_mix_frac'])*100:.0f}%)")
    print(f"   num_epochs    : {BASE_CONFIG['num_epochs']}  (train from scratch)")
    print(f"{'='*65}")

    config = {**BASE_CONFIG}
    # Build final config by merging BASE + experiment-specific keys
    config['num_assets']          = exp_cfg['num_assets']
    config['sub_day']             = exp_cfg['sub_day']
    config['pred_len']            = exp_cfg['pred_len']
    config['batch_size']          = exp_cfg['batch_size']
    config['beta_kl']             = exp_cfg['beta_kl']
    config['beta_anneal_epochs']  = exp_cfg['beta_anneal_epochs']
    config['gamma_sparsity']      = exp_cfg.get('gamma_sparsity', 0.0)
    config['nll_detach']          = exp_cfg.get('nll_detach', BASE_CONFIG['nll_detach'])
    config['spectral_config']      = {
        **BASE_CONFIG['spectral_config'],
        **exp_cfg.get('spectral_config', {})
    }
    config['training_rounds']     = exp_cfg['training_rounds']
    config['version']             = f'v2_multi_{dataset_name}'

    # real_train_datasets + real_test_datasets — both point to same dataset
    config['real_train_datasets'] = [dataset_name]  # ← activates MultivariateRealDataset
    config['real_test_datasets']  = [dataset_name]

    # Inject prior_mix_frac into prior_config
    config['prior_config'] = {
        **BASE_CONFIG['prior_config'],
        'prior_mix_frac': exp_cfg['prior_mix_frac'],
    }


    # Write config to file
    config_path = f'{PROJECT_ROOT}/core/config.{config["version"]}.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print(f"📝 Config saved: {config_path}")


    # =====================================================================
    # 1. Verify real validation datasets exist (pkl files)
    # =====================================================================
    REAL_VAL_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_val_datasets')
    os.makedirs(REAL_VAL_DIR, exist_ok=True)

    # Check which datasets are needed for real_test_datasets
    real_test_ds = config.get('real_test_datasets', [])
    padded = 'pad' if False else 'nopad'  # matches real_data_args.yaml pad=false
    MAX_LEN = 512

    missing = []
    for ds in real_test_ds:
        pkl_name = f'{ds}_{padded}_{MAX_LEN}.pkl'
        pkl_path = os.path.join(REAL_VAL_DIR, pkl_name)
        if os.path.exists(pkl_path):
            size_mb = os.path.getsize(pkl_path) / (1024*1024)
            print(f'  ✅ {ds:30s} → {pkl_name} ({size_mb:.1f} MB)')
        else:
            missing.append(ds)
            print(f'  ❌ {ds:30s} → MISSING')

    if missing:
        print(f'\n🔄 Generating {len(missing)} missing dataset(s) from GluonTS...')
        print('   This downloads from AWS and converts to pkl. May take 1-5 min per dataset.')
        !python data/scripts/store_real_datasets.py
        # Verify again
        for ds in missing:
            pkl_path = os.path.join(REAL_VAL_DIR, f'{ds}_{padded}_{MAX_LEN}.pkl')
            if os.path.exists(pkl_path):
                print(f'  ✅ {ds} → OK')
            else:
                print(f'  ⚠️  {ds} → STILL MISSING. Check store_real_datasets.py output above.')
    else:
        print(f'\n✅ All {len(real_test_ds)} real validation datasets found.')


    # =====================================================================
    # 2. RUN TRAINING in EXPERIMENTS setups
    # =====================================================================
    # Launch training — stream logs live
    cmd = ['python', f'{PROJECT_ROOT}/core/train.py', '-c', config_path]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.stdout.close()
    rc = proc.wait()

    if rc == 0:
        print(f"\n✅ {dataset_name} training complete.")
    else:
        print(f"\n❌ {dataset_name} training failed (exit code {rc}). Stopping.")
        break





🚀 CROSS-ASSET TRAINING: EXCHANGE_RATE
   num_assets    : 8
   prior_mix_frac: 0.5  (synthetic 50% + real 50%)
   num_epochs    : 30  (train from scratch)
📝 Config saved: /content/SC-Mamba/core/config.v2_multi_exchange_rate.yaml
  ✅ exchange_rate                  → exchange_rate_nopad_512.pkl (0.3 MB)

✅ All 1 real validation datasets found.
config:
{'batch_size': 8,
 'beta_anneal_epochs': 10,
 'beta_kl': 0.2,
 'context_len': 256,
 'continue_training': False,
 'debugging': False,
 'diag_prints': True,
 'encoding_dropout': 0.1,
 'gamma_sparsity': 0.1,
 'handle_constants_model': True,
 'initial_lr': 5e-05,
 'learning_rate': 1e-05,
 'loss': 'nll',
 'lr_scheduler': 'cosine',
 'max_seq_len': 256,
 'min_seq_len': 60,
 'model_prefix': '/content/drive/MyDrive/Colab '
                 'Notebooks/SCMamba/sc_mamba_checkpoints',
 'multipoint': True,
 'nll_detach': True,
 'num_assets': 8,
 'num_epochs': 30,
 'pred_len': 20,
 'pred_len_min': 10,
 'pred_len_sample': False,
 'prior_config': {'curricul

In [ ]:
!ls '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'

SCMamba_v2_multi_exchange_rate_best_mase.pth
SCMamba_v2_multi_exchange_rate_best.pth
SCMamba_v2_multi_exchange_rate_Final.pth
SCMamba_v2_multi_exchange_rate.pth


In [ ]:
# @title 11_eval_ckp_crossAsset
"""
11_eval_ckp_crossAsset.py
==============================
Ablation evaluation script: num_assets=1 (univariate) vs num_assets=8 (cross-asset graph).

Run directly via terminal:
python benchmark/01_Ckp_Eval_CrossAsset.py
"""

import sys
import os
import torch
import numpy as np
import pandas as pd
import yaml
from pprint import pprint

# Setup paths and environment
# SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
# PROJECT_ROOT = os.path.dirname(SCRIPT_DIR)
os.environ['TRITON_F32_DEFAULT'] = 'ieee'

from core.models import SCMamba_Forecaster
from core.real_data_val_pipeline import validate_on_real_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALER   = 'min_max'
CKPT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints' #os.path.join(PROJECT_ROOT, 'checkpoints')

# SSM config — must match your training config exactly
SSM_CONFIG = {
    'mamba2'             : True,
    'num_encoder_layers' : 2,
    'd_state'            : 128,
    'headdim'            : 128,
    'block_expansion'    : 2,
    'token_embed_len'    : 1024,
    'chunk_size'         : 256,
    'linear_seq'         : 15,
    'norm'               : True,
    'norm_type'          : 'layernorm',
    'residual'           : False,
    'global_residual'    : False,
    'bidirectional'      : False,
    'in_proj_norm'       : False,
    'enc_conv'           : True,
    'enc_conv_kernel'    : 5,
    'init_dil_conv'      : True,
    'init_conv_kernel'   : 5,
    'init_conv_max_dilation': 3,
    'initial_gelu_flag'  : True,
}

print('Setup complete. DEVICE =', DEVICE)


def load_model(ckpt_path: str, num_assets: int, ssm_config: dict) -> SCMamba_Forecaster:
    """Load SCMamba_Forecaster from checkpoint."""
    model = SCMamba_Forecaster(N_assets=num_assets, ssm_config=ssm_config).to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    # Support both raw state_dict and wrapped {'model_state_dict': ...}
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f'  ✅ Loaded: {os.path.basename(ckpt_path)} | N_assets={num_assets}')
    return model


def evaluate_model(
    model,
    dataset: str,
    scaler: str = 'min_max',
    sub_day: bool = False,
    label: str = 'model',
) -> dict:
    """
    Run validate_on_real_dataset and return a dict of metrics.
    Prints a formatted row suitable for comparison table.
    """
    print(f'\n🔍 Evaluating [{label}] on {dataset} ...')
    mase_, mae_, rmse_, smape_, nll_, crps_, mcrps_ = validate_on_real_dataset(
        dataset, model, DEVICE, scaler, subday=sub_day
    )
    result = {
        'label'  : label,
        'dataset': dataset,
        'MASE'   : round(mase_,  4),
        'MAE'    : round(mae_,   4),
        'RMSE'   : round(rmse_,  4),
        'SMAPE'  : round(smape_, 4),
        'NLL'    : round(nll_,   4),
        'mCRPS'  : round(mcrps_, 4),
    }
    mase_val = result.get("MASE")
    mcrps_val = result.get("mCRPS")

    mase_str = f"{mase_val:.4f}" if isinstance(mase_val, (int, float)) else "?"
    mcrps_str = f"{mcrps_val:.4f}" if isinstance(mcrps_val, (int, float)) else "?"

    print(f'  MASE={mase_str} | MAE={result.get("MAE","?")} | RMSE={result.get("RMSE","?")} | '
          f'SMAPE={result.get("SMAPE","?")} | NLL={result.get("NLL","?")} | mCRPS={mcrps_str}')
    return result


def main():
    # Checkpoint names based on your local folder
    ckpt_dir = CKPT_DIR

    ABLATION_CONFIGS = [
        {
            'label'      : 'N=1 (Univariate)',
            'ckpt'       : os.path.join(ckpt_dir, 'SCMamba_v_17data_N_uni_best_mase.pth'),
            'num_assets' : 1,
            'dataset'    : 'exchange_rate',
            'sub_day'    : False,
        },
        {
            'label'      : 'N=8 (Cross-Asset Graph)',
            'ckpt'       : os.path.join(ckpt_dir, 'SCMamba_v2_multi_exchange_rate_best_mase.pth'),
            'num_assets' : 8,
            'dataset'    : 'exchange_rate',
            'sub_day'    : False,
        },
        {
            'label'      : 'N=8 (Cross-Asset Graph) best NLL',
            'ckpt'       : os.path.join(ckpt_dir, 'SCMamba_v2_multi_exchange_rate_best.pth'),
            'num_assets' : 8,
            'dataset'    : 'exchange_rate',
            'sub_day'    : False,
        },
    ]

    results = []
    for cfg in ABLATION_CONFIGS:
        print(f'\n━━━━ {cfg["label"]} ━━━━')
        if not os.path.exists(cfg['ckpt']):
            print(f"⚠️ Warning: Checkpoint not found -> {cfg['ckpt']}")
            print("Skipping this configuration.\n")
            continue

        model = load_model(cfg['ckpt'], cfg['num_assets'], SSM_CONFIG)
        row   = evaluate_model(
            model,
            dataset = cfg['dataset'],
            scaler  = SCALER,
            sub_day = cfg['sub_day'],
            label   = cfg['label'],
        )
        results.append(row)

        # Free memory between runs
        del model
        torch.cuda.empty_cache()

    if not results:
        print("No valid results found to display.")
        return

    # ── Results Table + Interpretation ────────────────────────────────────────
    df = pd.DataFrame(results).set_index('label')
    print('\n' + '='*70)
    print('  ABLATION RESULTS: exchange_rate — N=1 vs N=8')
    print('='*70)
    # Show mCRPS instead of CRPS
    metric_cols = [c for c in ['MASE','MAE','RMSE','SMAPE','NLL','mCRPS'] if c in df.columns]
    print(df[metric_cols].to_string())
    print('='*70)
    print('  ↓ lower is better for all metrics')
    print()

    # Relative improvement (N=8 vs N=1)
    if len(df) >= 2:
        base  = df.iloc[0]   # N=1
        multi = df.iloc[1]   # N=8
        print('  Relative delta (N=8 - N=1) / |N=1|:')
        for m in metric_cols:
            delta_pct = (multi[m] - base[m]) / (abs(base[m]) + 1e-10) * 100
            arrow = '🟢' if delta_pct < 0 else '🔴'
            print(f'    {arrow}  {m:6s}: {delta_pct:+.1f}%')
        print()

# if __name__ == '__main__':
main()


Setup complete. DEVICE = cuda

━━━━ N=1 (Univariate) ━━━━
  ✅ Loaded: SCMamba_v_17data_N_uni_best_mase.pth | N_assets=1

🔍 Evaluating [N=1 (Univariate)] on exchange_rate ...
reading data from /content/SC-Mamba/data/real_val_datasets/exchange_rate_nopad_512.pkl
test 40
5
  MASE=1.5313 | MAE=0.010200000368058681 | RMSE=0.011300000362098217 | SMAPE=0.0064 | NLL=-3.4847 | mCRPS=0.0092

━━━━ N=8 (Cross-Asset Graph) ━━━━
  ✅ Loaded: SCMamba_v2_multi_exchange_rate_best_mase.pth | N_assets=8

🔍 Evaluating [N=8 (Cross-Asset Graph)] on exchange_rate ...
  [eval_pipeline] Multivariate aligned eval: N_assets=8 on exchange_rate
  MASE=0.2821 | MAE=0.00039999998989515007 | RMSE=0.0005000000237487257 | SMAPE=0.0008 | NLL=-4.4876 | mCRPS=0.0017

━━━━ N=8 (Cross-Asset Graph) best NLL ━━━━
  ✅ Loaded: SCMamba_v2_multi_exchange_rate_best.pth | N_assets=8

🔍 Evaluating [N=8 (Cross-Asset Graph) best NLL] on exchange_rate ...
  [eval_pipeline] Multivariate aligned eval: N_assets=8 on exchange_rate
  MASE=0.

# Zeroshot testing    `

In [ ]:
# @title 13_test_zeroshot_multi
"""
13_test_zeroshot_multi.py — v6 (Canonical Refactored)
=====================================================================
Zero-shot Foundation Model benchmark across GluonTS datasets.
Supports N=8 multivariate checkpoints with random sub-sampling.
=====================================================================
"""
import os, sys, yaml, warnings
import numpy as np, pandas as pd, torch
from pathlib import Path

# --- DIRECTORY SETUP (LOCAL/MAC ADAPTIVE) ---
# SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
# PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from scipy.stats import t as t_dist, norm as scipy_norm
from gluonts.dataset.repository.datasets import get_dataset
from gluonts.time_feature.seasonality import get_seasonality
from utilsforecast.losses import mase, mae, smape, rmse
from data.data_provider.multivariate_loader import MultivariateRealDataset
from core.eval_real_dataset import scale_data, nll_eval, crps_gaussian
from core.models import SCMamba_Forecaster

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONTEXT_LEN = 256
N_ASSETS    = 8
SCALER      = 'min_max'
SEEDS       = [7270, 860, 5390, 5191, 5734]

# Checkpoint path (Using the multi-asset best MASE checkpoint)
# colab_ckpt = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'
# CKPT_DIR = colab_ckpt if os.path.exists(colab_ckpt) else os.path.join(PROJECT_ROOT, 'checkpoints')
CHECKPOINT_PATH = os.path.join(CKPT_DIR, 'SCMamba_v2_multi_exchange_rate_best_mase.pth')

# All 17 Target datasets
TARGET_DATASETS = {
    "car_parts_without_missing": 12,
    "cif_2016":                  12,
    "covid_deaths":              30,
    "ercot":                     24,
    "exchange_rate":             30,
    "fred_md":                   12,
    "hospital":                  12,
    "m1_monthly":                18,
    "m1_quarterly":              8,
    "m3_monthly":                18,
    "m3_quarterly":              8,
    "nn5_daily_without_missing": 56,
    "nn5_weekly":                8,
    "tourism_monthly":           24,
    "tourism_quarterly":         8,
    "traffic":                   24,
    "weather":                   30,
}

# ─────────────────────────────────────────────────────────────────────────────
# Ensure all Datasets are generated (especially for Colab environments)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
REAL_VAL_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_val_datasets')
os.makedirs(REAL_VAL_DIR, exist_ok=True)
missing_ds = [ds for ds in TARGET_DATASETS if not os.path.exists(os.path.join(REAL_VAL_DIR, f'{ds}_nopad_512.pkl'))]

if missing_ds:
    print(f"\n🔄 Generating {len(missing_ds)} missing dataset(s) from GluonTS...")
    subprocess.run(['python', os.path.join(PROJECT_ROOT, 'data', 'scripts', 'store_real_datasets.py')])


# ─────────────────────────────────────────────────────────────────────────────
# Robust model loading
# ─────────────────────────────────────────────────────────────────────────────
def load_model_from_checkpoint(ckpt_path, device):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt)
    ssm_config = ckpt.get('ssm_config', {})

    # Inferred layers if missing
    if 'num_encoder_layers' not in ssm_config:
        layer_indices = sorted(set(int(k.split('.')[2]) for k in state_dict.keys() if 'mamba_encoder_layers.' in k))
        ssm_config['num_encoder_layers'] = max(layer_indices) + 1 if layer_indices else 2

    N_assets = ckpt.get('N_assets', N_ASSETS)
    model = SCMamba_Forecaster(N_assets=N_assets, ssm_config=ssm_config).to(device)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    print(f"  ✅ Loaded: {os.path.basename(ckpt_path)} | N_assets={model.N_assets}")
    return model

# ─────────────────────────────────────────────────────────────────────────────
# RobustZeroShotDataset for Asynchronous Coverage
# ─────────────────────────────────────────────────────────────────────────────
import pickle

class RobustZeroShotDataset(torch.utils.data.Dataset):
    """
    Local subclass specialized for Zero-Shot.
    1. Extracts specific `col_indices` (N=8).
    2. Performs 'Dense Time-Cropping': Finds the global min/max timestamp where AT LEAST ONE of the target series is active.
       (In purely asynchronous sets, requiring all 8 to be simultaneously active might yield 0 windows).
    3. Forward/Backward fills the remaining internal NaNs to ensure continuous signals for FFT.
    """
    def __init__(self, pkl_path: str, pred_len: int, context_len: int, split: str, col_indices: list, sub_day: bool = False):
        self.pred_len = pred_len
        self.context_len = context_len
        self.split = split
        self.N_assets = len(col_indices)
        self.sub_day = sub_day

        with open(pkl_path, 'rb') as f:
            df_raw = pickle.load(f)

        df_flat = df_raw.reset_index()
        df_piv = df_flat.pivot_table(index='date', columns='Series', values='target', aggfunc='first').sort_index()

        # Select only the specific 8 indices
        available = df_piv.shape[1]
        valid_idx = [i for i in col_indices if i < available]
        if len(valid_idx) < self.N_assets:
             raise ValueError(f"Requested {self.N_assets} assets, but only found {len(valid_idx)} valid indices.")

        df_sub = df_piv.iloc[:, valid_idx]

        # --- DENSE TIME-CROPPING & TELEMETRY ---
        # 1. Find the union of intervals where data is present
        # Drop rows where ALL 8 series are NaN (outside coverage)
        orig_len = len(df_sub)
        df_sub = df_sub.dropna(how='all')
        cropped_len = len(df_sub)

        # 2. Imputation Stats
        nan_count = df_sub.isna().sum().sum()
        total_cells = df_sub.size
        # Update log to intuitively explain the mathematical cropping process
        if total_cells > 0:
            print(f"      [RobustDS] Global Time Axis: {orig_len} -> 8-Asset Bounded Time: {cropped_len} (Trimmed {orig_len-cropped_len} empty edges). Inner NaNs (imputed): {nan_count}/{total_cells} ({nan_count/total_cells:.1%})")

        # Now, ffill/bfill for Spectral Soundness (FFT requires continuous signal)
        df_sub = df_sub.ffill().bfill().fillna(0.0)

        # Build Data Tensors
        ts_index = pd.to_datetime(df_sub.index)
        if sub_day:
            ts_feats = np.stack([ts_index.year.values, ts_index.month.values, ts_index.day.values, ts_index.day_of_week.values + 1, ts_index.day_of_year.values, ts_index.hour.values, ts_index.minute.values], axis=-1)
        else:
            ts_feats = np.stack([ts_index.year.values, ts_index.month.values, ts_index.day.values, ts_index.day_of_week.values + 1, ts_index.day_of_year.values], axis=-1)

        self.ts_feats = ts_feats.astype(np.float32)
        self.values = df_sub.values.astype(np.float32)
        # Store unpadded length to prevent metric corruption from zero-padding
        self.unpadded_len = cropped_len

        T_total = len(df_sub)
        n_test = pred_len
        min_train_required = context_len + pred_len
        max_val_allowed = max(pred_len, T_total - n_test - min_train_required)
        n_val = min(max(pred_len, 30), max_val_allowed)

        ideal_train_end = T_total - n_test - n_val
        train_end = max(ideal_train_end, min_train_required)
        if train_end > T_total:
            train_end = T_total

        if split == 'train':
            self._start, self._end = 0, train_end
            self.n_windows = max(0, train_end - context_len - pred_len + 1)
        elif split == 'val':
            val_target_start = T_total - n_test - n_val
            val_target_end   = T_total - n_test
            self._start = max(0, val_target_start - context_len)
            self._end = val_target_end
            self.n_windows = max(0, n_val - pred_len + 1)
        else: # test
            test_target_start = T_total - n_test
            self._start = max(0, test_target_start - context_len)
            self._end = T_total
            self.n_windows = max(0, n_test - pred_len + 1)

        self._split = split
        self._val_target_start = T_total - n_test - n_val
        self._test_target_start = T_total - n_test

    def __len__(self) -> int:
        return self.n_windows

    def __getitem__(self, idx: int) -> dict:
        if self._split == 'train':
            abs_start = self._start + idx
            ctx_end = abs_start + self.context_len
        else:
            target_start = (self._val_target_start if self._split == 'val' else self._test_target_start) + idx
            abs_start = max(0, target_start - self.context_len)
            ctx_end = target_start

        tgt_end = ctx_end + self.pred_len
        ctx_len_actual = ctx_end - abs_start

        if ctx_len_actual < self.context_len:
            pad = self.context_len - ctx_len_actual
            x = np.concatenate([np.zeros((pad, self.N_assets), dtype=np.float32), self.values[abs_start : ctx_end]], axis=0)
            ts_x = np.concatenate([np.zeros((pad, self.ts_feats.shape[1]), dtype=np.float32), self.ts_feats[abs_start : ctx_end]], axis=0)
        else:
            x = self.values[abs_start : ctx_end]
            ts_x = self.ts_feats[abs_start : ctx_end]

        y = self.values[ctx_end : tgt_end]
        ts_y = self.ts_feats[ctx_end : tgt_end]

        return {
            'x': torch.from_numpy(x), 'y': torch.from_numpy(y),
            'ts_x': torch.from_numpy(ts_x), 'ts_y': torch.from_numpy(ts_y),
            'window_idx': abs_start,
        }

# ─────────────────────────────────────────────────────────────────────────────
# Helper: canonical inference with subsetting
# ─────────────────────────────────────────────────────────────────────────────
def canonical_evaluate(
    model, pkl_path, pred_len, context_len, col_indices, scaler, device, sub_day=False,
):
    N = len(col_indices)

    # Use the Robust dataset that gracefully aligns asynchronous series via dense time-cropping
    try:
        test_ds = RobustZeroShotDataset(pkl_path, pred_len=pred_len, context_len=context_len, split='test', col_indices=col_indices, sub_day=sub_day)
        train_ds = RobustZeroShotDataset(pkl_path, pred_len=pred_len, context_len=context_len, split='train', col_indices=col_indices, sub_day=sub_day)
    except Exception as e:
        print(f"      [Robust Loader Error]: {e}")
        return None

    if len(test_ds) == 0:
        print(f"      [Not enough overlap windows] Found 0 windows after cropping.")
        return None

    ds_name = os.path.basename(pkl_path).replace('_nopad_512.pkl', '')
    try:
        gts_ds = get_dataset(ds_name, regenerate=False)
        seasonality = get_seasonality(gts_ds.metadata.freq)
        if gts_ds.metadata.freq == 'D': seasonality = 7
    except: seasonality = 1

    batch_train_dfs, batch_pred_dfs = [], []

    with torch.no_grad():
        for win_idx in range(len(test_ds)):
            sample = test_ds[win_idx]
            x, y = sample['x'].to(device), sample['y'].to(device)
            ts_x, ts_y = sample['ts_x'].to(device), sample['ts_y'].to(device)

            T_ctx, T_pred = x.shape[0], y.shape[0]
            data = {
                'history': x.permute(1, 0),
                'ts': ts_x.unsqueeze(0).expand(N, -1, -1),
                'target_dates': ts_y.unsqueeze(0).expand(N, -1, -1),
                'task': torch.zeros(N, T_pred, dtype=torch.int32, device=device),
            }

            output = model(data, prediction_length=T_pred)
            scaled_mu, scaled_sigma2 = scale_data(output, scaler)
            mu_np, sig_np, y_np = scaled_mu.cpu().numpy(), scaled_sigma2.cpu().numpy(), y.cpu().numpy()

            for asset_i in range(N):
                asset_id = f"zs_asset_{asset_i}"

                # IMPROVEMENT: Use the full unpadded history for MASE denominator calculation
                # instead of just the last window (which might be mostly zero-padding)
                train_hist_i = train_ds.values[:, asset_i]

                batch_train_dfs.append(pd.DataFrame({'id': [asset_id]*len(train_hist_i), 'target': train_hist_i}))

                sigma_i = np.sqrt(np.clip(sig_np[asset_i], 1e-6, None))
                crps_vals = crps_gaussian(mu_np[asset_i], sigma_i, y_np[:, asset_i])

                batch_pred_dfs.append(pd.DataFrame({
                    'id': [asset_id]*T_pred, 'pred': mu_np[asset_i], 'target': y_np[:, asset_i],
                    'variance': sig_np[asset_i], 'nll': nll_eval(torch.tensor(mu_np[asset_i]), torch.tensor(sig_np[asset_i]), torch.tensor(y_np[:, asset_i])).numpy(),
                    'crps': crps_vals
                }))

    train_df = pd.concat(batch_train_dfs)
    pred_df  = pd.concat(batch_pred_dfs)

    # MASE, CRPS, mCRPS (Safeguarded against NaN from 0 variance/flatlines)
    # utilsforecast's mase can divide by 0 if the naive training error is exactly 0.0 (e.g. constant ffill).
    # We add a tiny epsilon to avoid inf/nan.
    try:
        # Add epsilon to denominator to be absolutely sure we never divide by zero
        # utilsforecast doesn't provide epsilon natively, so we ensure train_df target is not all-zero
        mase_res = mase(pred_df, ['pred'], seasonality, train_df, 'id', 'target')
        # If MASE is inf/nan, fallback to MAE / (mean_abs_train + epsilon)
        mase_series = mase_res['pred'].replace([np.inf, -np.inf], np.nan)

        if mase_series.isna().any():
            for idx, val in mase_series.items():
                if np.isnan(val):
                    # Manual robust calculation for this ID
                    tdf_id = train_df[train_df['id'] == mase_res.loc[idx, 'id']]
                    pdf_id = pred_df[pred_df['id'] == mase_res.loc[idx, 'id']]
                    mae_id = np.mean(np.abs(pdf_id['target'].values - pdf_id['pred'].values))
                    y_train_id = tdf_id['target'].values
                    if len(y_train_id) > seasonality:
                        denom = np.mean(np.abs(y_train_id[seasonality:] - y_train_id[:-seasonality]))
                    else:
                        denom = 0.0 # Force fallback

                    if np.isnan(denom) or denom < 1e-8:
                        denom = np.mean(np.abs(y_train_id)) + 1e-8
                    mase_series.at[idx] = mae_id / denom

        mase_mean = float(mase_series.mean(skipna=True))
        # fallback if everything is nan
        if np.isnan(mase_mean): mase_mean = float('nan')
    except Exception as e:
        print(f"      [MASE Error]: {e}")
        mase_mean = float('nan')

    raw_crps = float(pred_df['crps'].replace([np.inf, -np.inf], np.nan).mean(skipna=True))
    mean_abs_target = float(train_df['target'].abs().mean(skipna=True)) if train_df['target'].abs().mean(skipna=True) > 0 else 1.0
    mcrps = raw_crps / mean_abs_target

    return {
        'mase': mase_mean, 'mae': float(mae(pred_df, ['pred'], 'id', 'target')['pred'].mean()),
        'rmse': float(rmse(pred_df, ['pred'], 'id', 'target')['pred'].mean()),
        'smape': float(smape(pred_df, ['pred'], 'id', 'target')['pred'].mean()),
        'nll': float(pred_df['nll'].mean()), 'crps': raw_crps, 'mcrps': mcrps
    }

def get_total_assets(pkl_path):
    with open(pkl_path, 'rb') as f:
        df = pickle.load(f)
    return len(df.index.get_level_values('Series').unique())

def benchmark_dataset(ds_name, pred_len, model, device, seeds):
    # project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
    pkl_path = os.path.join(PROJECT_ROOT, 'data', 'real_val_datasets', f'{ds_name}_nopad_512.pkl')
    if not os.path.exists(pkl_path):
        print(f"  ⏭️ Skip: {ds_name} (PKL missing)")
        return None

    seed_results = []
    try:
        total_assets = get_total_assets(pkl_path)
    except Exception as e:
         print(f"  ⏭️ Skip: {ds_name} (Metadata read error: {e})")
         return None

    for seed in seeds:
        rng = np.random.default_rng(seed)

        # Intersection Retry Logic: Try to find a valid subset of 8 assets
        max_retries = 10
        valid_res = None

        for attempt in range(max_retries):
            try:
                col_idx = sorted(rng.choice(total_assets, size=N_ASSETS, replace=False)) if total_assets >= N_ASSETS else list(range(total_assets))
                valid_res = canonical_evaluate(model, pkl_path, pred_len, CONTEXT_LEN, col_idx, SCALER, device)
                if valid_res is not None:
                    break # Success!
            except Exception as e:
                pass # Try another slice

        if valid_res:
            seed_results.append(valid_res)
        else:
            print(f"    Seed {seed}: Failed to find valid overlapping sequence after {max_retries} retries.")

    if not seed_results: return None

    # Mean across seeds
    mase_m = np.nanmean([r['mase'] for r in seed_results])
    mcrps_m = np.nanmean([r['mcrps'] for r in seed_results])
    print(f"  📡 {ds_name:25s} | MASE={mase_m:.4f} | mCRPS={mcrps_m:.4f} (Avg across {len(seed_results)} seeds)")

    return {
        'dataset': ds_name, 'mase': mase_m, 'mcrps': mcrps_m,
        'mae': np.nanmean([r['mae'] for r in seed_results]),
        'nll': np.nanmean([r['nll'] for r in seed_results]),
        'count': len(seed_results)
    }

def main():
    print(f"\n{'='*70}\n🚀 SC-Mamba Multi-Asset Zero-Shot Benchmark\n{'='*70}")

    try:
        model = load_model_from_checkpoint(CHECKPOINT_PATH, DEVICE)
    except Exception as e:
        print(f"❌ Load error: {e}")
        return

    results = []
    for ds, pl in TARGET_DATASETS.items():
        res = benchmark_dataset(ds, pl, model, DEVICE, SEEDS)
        if res: results.append(res)

    if not results:
        print("No results generated.")
        return

    df = pd.DataFrame(results)
    print(f"\n{'='*85}")
    print(f"{'Dataset':<30} {'MASE':>10} {'mCRPS':>10} {'MAE':>10} {'NLL':>10}")
    print(f"{'─'*85}")
    for _, r in df.iterrows():
        print(f"{r['dataset']:<30} {r['mase']:>10.4f} {r['mcrps']:>10.4f} {r['mae']:>10.4f} {r['nll']:>10.4f}")
    print(f"{'─'*85}")
    print(f"💎 Avg MASE: {df['mase'].mean():.4f} | Avg mCRPS: {df['mcrps'].mean():.4f}")
    print(f"{'='*85}\n✅ Zero-shot Multi complete.")

# if __name__ == "__main__":
main()




🚀 SC-Mamba Multi-Asset Zero-Shot Benchmark
  ✅ Loaded: SCMamba_v2_multi_exchange_rate_best_mase.pth | N_assets=8
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
      [RobustDS] Global Time Axis: 51 -> 8-Asset Bounded Time: 51 (Trimmed 0 empty edges). Inner NaNs (imputed): 0/408 (0.0%)
    